# Variability Analysis Figures

Visualisations for:
1. PRE vs POST variance by Light group
2. Baseline imbalances (3-way interaction)
3. Bootstrap variance ratios
4. Residual SD comparison (PI's approach)
5. IS variance pattern (ISF decreased, CTR increased)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', context='talk', font_scale=0.95)
PAL_LIGHT = {'CTR': '#4C72B0', 'ISF': '#DD8452'}
PAL_PERIOD = {'PRE': '#66c2a5', 'POST': '#fc8d62'}

def clean_colnames(df):
    df = df.copy()
    df.columns = (df.columns.astype(str).str.strip()
        .str.replace(r'[^\w]+', '_', regex=True)
        .str.replace(r'_+', '_', regex=True).str.strip('_'))
    return df

circ = clean_colnames(pd.read_csv('Circadian_raw.csv'))
circ['ID'] = pd.to_numeric(circ['ID'], errors='coerce').astype('Int64')
circ['PRE_POST'] = circ['PRE_POST'].astype(str)
circ['Light_new'] = circ['Light_new'].astype(str).str.strip()
circ['Age_new'] = circ['Age_new'].astype(str).str.strip()
circ['Sex_new'] = circ['Sex_new'].astype(str).str.strip()

metrics = ['IS', 'IV', 'RA', 'Amplitude']
print(f'Loaded {circ["ID"].nunique()} mice')

## Figure 1: PRE vs POST Spread by Light Group
Strip + box plots showing the raw distributions. Visual check for variance differences.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, metric in enumerate(metrics):
    for j, light in enumerate(['CTR', 'ISF']):
        ax = axes[j, i]
        d = circ[circ['Light_new'] == light][['PRE_POST', metric]].dropna()
        
        sns.boxplot(data=d, x='PRE_POST', y=metric, order=['PRE', 'POST'],
                    palette=PAL_PERIOD, width=0.5, fliersize=0,
                    boxprops=dict(alpha=0.3), ax=ax)
        sns.stripplot(data=d, x='PRE_POST', y=metric, order=['PRE', 'POST'],
                      palette=PAL_PERIOD, jitter=0.25, alpha=0.6, size=5, ax=ax)
        
        # Annotate SD
        for k, period in enumerate(['PRE', 'POST']):
            vals = d[d['PRE_POST'] == period][metric]
            sd = vals.std(ddof=1)
            ax.text(k, ax.get_ylim()[1] * 0.95, f'SD={sd:.3f}',
                    ha='center', fontsize=9, fontstyle='italic')
        
        ax.set_title(f'{metric} — {light}', fontsize=12)
        ax.set_xlabel('')
        if i > 0:
            ax.set_ylabel('')

axes[0, 0].text(-0.35, 0.5, 'CTR', transform=axes[0, 0].transAxes,
               fontsize=14, fontweight='bold', va='center', rotation=90)
axes[1, 0].text(-0.35, 0.5, 'ISF', transform=axes[1, 0].transAxes,
               fontsize=14, fontweight='bold', va='center', rotation=90)

fig.suptitle('PRE vs POST Distributions by Light Group (with SD annotated)',
             fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig('figures/fig_var_pre_post_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 2: Variance Change — ISF vs CTR
Bar chart showing the variance (SD²) at PRE and POST for each Light group, with arrows indicating direction of change.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    bars = []
    for light in ['CTR', 'ISF']:
        for period in ['PRE', 'POST']:
            vals = circ[(circ['Light_new'] == light) & (circ['PRE_POST'] == period)][metric].dropna()
            bars.append({'Light': light, 'Period': period, 'var': vals.var(ddof=1), 'sd': vals.std(ddof=1)})
    
    bdf = pd.DataFrame(bars)
    x = np.arange(2)  # CTR, ISF
    width = 0.35
    
    pre_vals = bdf[bdf['Period'] == 'PRE']['sd'].values
    post_vals = bdf[bdf['Period'] == 'POST']['sd'].values
    
    ax.bar(x - width/2, pre_vals, width, label='PRE', color=PAL_PERIOD['PRE'], alpha=0.8)
    ax.bar(x + width/2, post_vals, width, label='POST', color=PAL_PERIOD['POST'], alpha=0.8)
    
    # Add arrows showing direction
    for k in range(2):
        change = post_vals[k] - pre_vals[k]
        pct = change / pre_vals[k] * 100
        color = '#2ca02c' if change < 0 else '#d62728'
        symbol = '\u2193' if change < 0 else '\u2191'  # down/up arrow
        ax.text(k, max(pre_vals[k], post_vals[k]) + ax.get_ylim()[1] * 0.03,
                f'{symbol}{abs(pct):.1f}%', ha='center', fontsize=10,
                color=color, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(['CTR', 'ISF'])
    ax.set_ylabel('Standard Deviation' if i == 0 else '')
    ax.set_title(metric, fontsize=13)
    if i == 0:
        ax.legend(fontsize=9)

fig.suptitle('Standard Deviation: PRE vs POST by Light Group',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_var_sd_change_by_light.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3: IS Variance — The Suggestive Pattern
ISF mice became more homogeneous while CTR mice became more variable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: IS distributions PRE vs POST, faceted by Light
ax = axes[0]
d = circ[['PRE_POST', 'Light_new', 'IS']].dropna()
d['group'] = d['Light_new'] + ' ' + d['PRE_POST']
order = ['CTR PRE', 'CTR POST', 'ISF PRE', 'ISF POST']
colors = [PAL_PERIOD['PRE'], PAL_PERIOD['POST'], PAL_PERIOD['PRE'], PAL_PERIOD['POST']]

parts = ax.violinplot(
    [d[d['group'] == g]['IS'].values for g in order],
    positions=[0, 1, 2.5, 3.5],
    showmeans=True, showmedians=False
)
for pc, color in zip(parts['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.6)
parts['cmeans'].set_color('black')

# Annotate variance
for pos, g in zip([0, 1, 2.5, 3.5], order):
    v = d[d['group'] == g]['IS'].var(ddof=1)
    ax.text(pos, -0.05, f'var={v:.4f}', ha='center', fontsize=8, fontstyle='italic')

ax.set_xticks([0, 1, 2.5, 3.5])
ax.set_xticklabels(['CTR\nPRE', 'CTR\nPOST', 'ISF\nPRE', 'ISF\nPOST'], fontsize=10)
ax.set_ylabel('IS')
ax.set_title('A. IS distributions', fontsize=13)

# Panel B: Paired PRE->POST IS trajectories by Light
ax = axes[1]
for light, color in PAL_LIGHT.items():
    sub = circ[circ['Light_new'] == light]
    pre = sub[sub['PRE_POST'] == 'PRE'].set_index('ID')['IS']
    post = sub[sub['PRE_POST'] == 'POST'].set_index('ID')['IS']
    common = pre.index.intersection(post.index)
    for mid in common:
        offset = -0.05 if light == 'CTR' else 0.05
        ax.plot([0 + offset, 1 + offset], [pre[mid], post[mid]],
                color=color, alpha=0.15, lw=0.8)
    # Group means
    ax.plot([0 + (-0.05 if light=='CTR' else 0.05),
             1 + (-0.05 if light=='CTR' else 0.05)],
            [pre.loc[common].mean(), post.loc[common].mean()],
            color=color, lw=3, marker='o', markersize=10, label=light, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(['PRE', 'POST'])
ax.set_ylabel('IS')
ax.set_title('B. Individual IS trajectories', fontsize=13)
ax.legend(fontsize=10)

# Panel C: Bootstrap variance ratio
ax = axes[2]
rng = np.random.default_rng(42)
n_boot = 5000

for k, (light, color) in enumerate(PAL_LIGHT.items()):
    pre = circ[(circ['PRE_POST'] == 'PRE') & (circ['Light_new'] == light)]['IS'].dropna().values
    post = circ[(circ['PRE_POST'] == 'POST') & (circ['Light_new'] == light)]['IS'].dropna().values
    
    ratios = []
    for _ in range(n_boot):
        bp = rng.choice(pre, size=len(pre), replace=True)
        bq = rng.choice(post, size=len(post), replace=True)
        vp = bp.var(ddof=1)
        if vp > 0:
            ratios.append(bq.var(ddof=1) / vp)
    
    ratios = np.array(ratios)
    ax.hist(ratios, bins=50, alpha=0.5, color=color, label=f'{light} (median={np.median(ratios):.2f})',
            density=True, edgecolor='white')

ax.axvline(1.0, color='red', ls='--', lw=2, label='No change (ratio=1)')
ax.set_xlabel('Variance ratio (POST / PRE)')
ax.set_ylabel('Density')
ax.set_title('C. Bootstrap IS variance ratio', fontsize=13)
ax.legend(fontsize=9)

fig.suptitle('IS Variance Pattern: ISF Mice Became More Homogeneous, CTR Did Not',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_var_IS_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 4: Baseline Imbalance — 3-Way Interaction (Age x Sex x Light)
Shows that the significant interactions at PRE are driven by small Young subgroups.

In [ ]:
pre = circ[circ['PRE_POST'] == 'PRE'].copy()
post = circ[circ['PRE_POST'] == 'POST'].copy()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

age_order = ['Young', 'Mid', 'Old']

for i, metric in enumerate(metrics):
    # PRE
    ax = axes[0, i]
    sns.pointplot(data=pre, x='Age_new', y=metric, hue='Light_new',
                  order=age_order, palette=PAL_LIGHT, errorbar='se',
                  dodge=0.15, markers=['o', 's'], capsize=0.05, ax=ax)
    # Annotate n per group
    for j, age in enumerate(age_order):
        for light in ['CTR', 'ISF']:
            n = len(pre[(pre['Age_new'] == age) & (pre['Light_new'] == light)][metric].dropna())
            offset = -0.08 if light == 'CTR' else 0.08
            ax.text(j + offset, ax.get_ylim()[0], f'n={n}', ha='center', fontsize=7, alpha=0.6)
    ax.set_title(f'{metric} (PRE)', fontsize=12)
    ax.set_xlabel('')
    if i > 0:
        ax.set_ylabel('')
        ax.legend().remove()
    else:
        ax.legend(title='Light', fontsize=8, title_fontsize=9)

    # POST
    ax = axes[1, i]
    sns.pointplot(data=post, x='Age_new', y=metric, hue='Light_new',
                  order=age_order, palette=PAL_LIGHT, errorbar='se',
                  dodge=0.15, markers=['o', 's'], capsize=0.05, ax=ax)
    for j, age in enumerate(age_order):
        for light in ['CTR', 'ISF']:
            n = len(post[(post['Age_new'] == age) & (post['Light_new'] == light)][metric].dropna())
            offset = -0.08 if light == 'CTR' else 0.08
            ax.text(j + offset, ax.get_ylim()[0], f'n={n}', ha='center', fontsize=7, alpha=0.6)
    ax.set_title(f'{metric} (POST)', fontsize=12)
    ax.set_xlabel('Age Group')
    if i > 0:
        ax.set_ylabel('')
        ax.legend().remove()
    else:
        ax.legend(title='Light', fontsize=8, title_fontsize=9)

axes[0, 0].text(-0.35, 0.5, 'PRE', transform=axes[0, 0].transAxes,
               fontsize=14, fontweight='bold', va='center', rotation=90)
axes[1, 0].text(-0.35, 0.5, 'POST', transform=axes[1, 0].transAxes,
               fontsize=14, fontweight='bold', va='center', rotation=90)

fig.suptitle('Circadian Metrics by Age x Light: PRE vs POST\n(n per cell annotated — note Young n=2-3)',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_var_baseline_imbalance.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 5: Baseline Imbalance — Sex x Light (Amplitude)
The significant Sex x Light interaction at PRE for Amplitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for j, (period, data) in enumerate([('PRE', pre), ('POST', post)]):
    ax = axes[j]
    sns.boxplot(data=data, x='Sex_new', y='Amplitude', hue='Light_new',
                palette=PAL_LIGHT, width=0.6, fliersize=0,
                boxprops=dict(alpha=0.3), ax=ax)
    sns.stripplot(data=data, x='Sex_new', y='Amplitude', hue='Light_new',
                  palette=PAL_LIGHT, dodge=True, jitter=0.15, alpha=0.6, size=5, ax=ax)
    
    ax.set_title(f'Amplitude ({period})', fontsize=13)
    ax.set_xlabel('Sex')
    ax.set_ylabel('Amplitude' if j == 0 else '')
    
    # Remove duplicate legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title='Light', fontsize=9)

fig.suptitle('Sex x Light Interaction for Amplitude\n(Significant at PRE [p=0.014], disappears at POST)',
             fontsize=14, fontweight='bold', y=1.03)
fig.tight_layout()
fig.savefig('figures/fig_var_sex_light_amplitude.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 6: Residual SD — PI's Approach Formalised
Residual SD from separate OLS models (Metric ~ Light + Age + Sex) fit to PRE and POST.

In [ ]:
# Compute residual SDs
resid_rows = []
for metric in metrics:
    for period in ['PRE', 'POST']:
        d = circ[circ['PRE_POST'] == period][['Light_new', 'Age_new', 'Sex_new', metric]].dropna()
        m = smf.ols(f'{metric} ~ C(Light_new) + C(Age_new) + C(Sex_new)', data=d).fit()
        resid_rows.append({'Metric': metric, 'Period': period, 'residual_sd': np.sqrt(m.mse_resid)})

rsd = pd.DataFrame(resid_rows)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(metrics))
width = 0.35

pre_sd = rsd[rsd['Period'] == 'PRE']['residual_sd'].values
post_sd = rsd[rsd['Period'] == 'POST']['residual_sd'].values

bars1 = ax.bar(x - width/2, pre_sd, width, label='PRE', color=PAL_PERIOD['PRE'], alpha=0.8)
bars2 = ax.bar(x + width/2, post_sd, width, label='POST', color=PAL_PERIOD['POST'], alpha=0.8)

# Annotate % change
for i in range(len(metrics)):
    pct = (post_sd[i] - pre_sd[i]) / pre_sd[i] * 100
    color = '#2ca02c' if pct < 0 else '#d62728'
    symbol = '\u2193' if pct < 0 else '\u2191'
    ax.text(i, max(pre_sd[i], post_sd[i]) + 0.02, f'{symbol}{abs(pct):.1f}%',
            ha='center', fontsize=11, color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Residual SD (from OLS: Metric ~ Light + Age + Sex)')
ax.set_title('Residual Variability: PRE vs POST\n(After controlling for Light, Age, Sex)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

fig.tight_layout()
fig.savefig('figures/fig_var_residual_sd.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 7: Updated Model — Adding Interactions Does Not Change Results
Side-by-side p-values from original vs updated LME.

In [ ]:
# Run both models
results = []
circ_m = circ.copy()
circ_m['PRE_POST'] = circ_m['PRE_POST'].astype('category').cat.set_categories(['PRE','POST'])
circ_m['Light_new'] = circ_m['Light_new'].astype('category').cat.set_categories(['CTR','ISF'])
circ_m['Age_new'] = circ_m['Age_new'].astype('category')
circ_m['Sex_new'] = circ_m['Sex_new'].astype('category')

for metric in metrics:
    d = circ_m[['ID','PRE_POST','Light_new','Age_new','Sex_new',metric]].dropna()
    
    m1 = smf.mixedlm(f'{metric} ~ PRE_POST * Light_new + Age_new + Sex_new',
                      data=d, groups=d['ID']).fit(method='lbfgs', reml=True)
    m2 = smf.mixedlm(f'{metric} ~ PRE_POST * Light_new + Age_new * Light_new + Sex_new * Light_new',
                      data=d, groups=d['ID']).fit(method='lbfgs', reml=True)
    
    t1 = [t for t in m1.pvalues.index if 'PRE_POST' in t and 'Light' in t]
    t2 = [t for t in m2.pvalues.index if 'PRE_POST' in t and 'Light' in t]
    
    results.append({
        'Metric': metric,
        'p_original': float(m1.pvalues[t1[0]]) if t1 else np.nan,
        'p_updated': float(m2.pvalues[t2[0]]) if t2 else np.nan,
    })

rdf = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, rdf['p_original'], width, label='Original\n(+ Age + Sex)', color='#4C72B0', alpha=0.8)
ax.bar(x + width/2, rdf['p_updated'], width, label='Updated\n(+ Age*Light + Sex*Light)', color='#DD8452', alpha=0.8)

ax.axhline(0.05, color='red', ls='--', lw=1.5, label='p = 0.05')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('p-value (PRE_POST x Light interaction)')
ax.set_title('Adding Baseline Interactions Does Not Change Treatment Conclusions',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_ylim(0, 1.05)

# Annotate p-values
for i in range(len(metrics)):
    ax.text(i - width/2, rdf['p_original'].iloc[i] + 0.02, f'{rdf["p_original"].iloc[i]:.3f}',
            ha='center', fontsize=9)
    ax.text(i + width/2, rdf['p_updated'].iloc[i] + 0.02, f'{rdf["p_updated"].iloc[i]:.3f}',
            ha='center', fontsize=9)

fig.tight_layout()
fig.savefig('figures/fig_var_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

1. **PRE vs POST distributions** (Fig 1-2): Visual variance differences are modest; only RA shows a clear decrease
2. **IS variance pattern** (Fig 3): ISF variance decreased 34%, CTR increased 32% — suggestive but not statistically significant (bootstrap CI includes 0)
3. **Baseline imbalances** (Fig 4-5): Significant 3-way interactions at PRE driven by Young subgroups (n=2-3 per cell); disappear at POST
4. **Residual SD** (Fig 6): RA residual SD decreased 12% from PRE to POST; other metrics showed minimal change
5. **Model robustness** (Fig 7): Adding Age*Light and Sex*Light interactions to the LME produces identical treatment p-values